# BeeWatch — Sensor Autoencoder Training
**Bagian:** Model Sensor  
**Framework:** TensorFlow / Keras  
**Dataset:** Kaggle Beehive Sounds (`all_data_updated.csv`)  
**Fitur input:** `hive temp`, `hive humidity`, `hive pressure` (3 fitur, tanpa hive weight)

---
### Alur Notebook ini
1. Load & eksplorasi dataset
2. Pilih dan bersihkan fitur sensor
3. Split data → Normalisasi (scaler fit hanya di train)
4. Bangun & latih Sensor Autoencoder (Keras)
5. Hitung reconstruction error & tentukan threshold
6. Evaluasi visual
7. Simpan artefak: model + scaler + threshold

## 1. Setup & Import

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import json
import os

import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

print(f"TensorFlow  : {tf.__version__}")
print(f"GPU tersedia: {len(tf.config.list_physical_devices('GPU')) > 0}")

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

os.makedirs('models', exist_ok=True)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

## 2. Load Dataset

In [ ]:
df = pd.read_csv('data/sensor_data/all_data_updated.csv')

print(f"Shape dataset : {df.shape}")
print(f"Kolom         : {df.columns.tolist()}")
print()
df.head()

## 3. Eksplorasi Data Sensor

Kita hanya pakai 3 kolom: `hive temp`, `hive humidity`, `hive pressure`.  
Kolom `target` **tidak dipakai saat training** (unsupervised), tapi berguna untuk evaluasi di akhir.

In [ ]:
SENSOR_FEATURES = ['hive temp', 'hive humidity', 'hive pressure']
INPUT_DIM       = len(SENSOR_FEATURES)  # 3

print("=== Missing Values ===")
print(df[SENSOR_FEATURES].isnull().sum())
print()
print("=== Statistik Deskriptif ===")
print(df[SENSOR_FEATURES].describe().round(3))
print()
print("=== Distribusi Target (tidak dipakai training, hanya referensi) ===")
print(df['target'].value_counts().sort_index())

In [ ]:
# Visualisasi distribusi tiap fitur sensor
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ['steelblue', 'seagreen', 'tomato']
labels = ['Suhu (°C)', 'Kelembaban (%)', 'Tekanan (hPa)']

for i, (feat, label, color) in enumerate(zip(SENSOR_FEATURES, labels, colors)):
    axes[i].hist(df[feat], bins=40, color=color, alpha=0.8, edgecolor='white')
    axes[i].set_title(f'Distribusi: {label}', fontsize=12)
    axes[i].set_xlabel(label)
    axes[i].set_ylabel('Frekuensi')

plt.suptitle('Distribusi Fitur Sensor (Seluruh Dataset)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('sensor_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Lihat rata-rata sensor per kelas target (untuk referensi, tidak mempengaruhi training)
print("Rata-rata nilai sensor per kelas target:")
print(df.groupby('target')[SENSOR_FEATURES].mean().round(2))

## 4. Persiapan Data

### Strategi pemilihan data "normal" untuk training

Dataset ini punya kolom `target` (0–5) yang merepresentasikan kondisi koloni.  
Karena kita pakai **unsupervised autoencoder**, kita punya dua pilihan:

- **Opsi A** — Pakai semua data (model belajar pola keseluruhan dataset)
- **Opsi B** — Pakai hanya kelas mayoritas sebagai "normal", anggap kelas lain sebagai anomali

Di sini kita pakai **Opsi A** karena:
1. Kita tidak punya definisi pasti kelas mana yang "normal" vs "anomali" untuk sensor
2. Distribusi sensor antar kelas tidak berbeda drastis
3. Autoencoder akan belajar pola statistik keseluruhan dataset

In [ ]:
# Ambil fitur sensor (tidak ada NaN, jadi tidak perlu dropna)
X_sensor = df[SENSOR_FEATURES].values
print(f"Shape data sensor: {X_sensor.shape}")
print(f"Total sampel     : {X_sensor.shape[0]}")

In [ ]:
# ============================================================
# PENTING: Split DULU, baru normalisasi
# Scaler hanya boleh fit di data training
# ============================================================

X_train_raw, X_val_raw = train_test_split(
    X_sensor,
    test_size=0.15,
    random_state=42
)

print(f"Training samples  : {X_train_raw.shape[0]}")
print(f"Validation samples: {X_val_raw.shape[0]}")

In [ ]:
# Normalisasi: fit HANYA di data train, transform ke val
sensor_scaler = StandardScaler()

X_train = sensor_scaler.fit_transform(X_train_raw)  # belajar dari train
X_val   = sensor_scaler.transform(X_val_raw)         # terapkan ke val

print("Setelah normalisasi (train):")
print(f"  Mean per fitur : {X_train.mean(axis=0).round(4)}  (harusnya ~0)")
print(f"  Std  per fitur : {X_train.std(axis=0).round(4)}   (harusnya ~1)")
print()
print("Scaler parameters (disimpan untuk deploy):")
for feat, mean_, scale_ in zip(SENSOR_FEATURES, sensor_scaler.mean_, sensor_scaler.scale_):
    print(f"  {feat:20s}: mean={mean_:.4f}, std={scale_:.4f}")

## 5. Arsitektur Model

```
Input (3)  →  Dense(16) → ReLU → BN  →  Bottleneck(2) → ReLU
           →  Dense(16) → ReLU → BN  →  Output(3)
```

Bottleneck **2** dari input **3** → model dipaksa menemukan hubungan antar ketiga sensor.

In [ ]:
ENCODING_DIM = 2

def build_sensor_autoencoder(input_dim: int, encoding_dim: int) -> keras.Model:
    """
    Sensor Autoencoder untuk BeeWatch.
    
    Encoder : input_dim → 16 → encoding_dim
    Decoder : encoding_dim → 16 → input_dim
    
    Args:
        input_dim   : jumlah fitur sensor (3)
        encoding_dim: ukuran bottleneck (2)
    """
    inputs = keras.Input(shape=(input_dim,), name='sensor_input')

    # --- Encoder ---
    x = keras.layers.Dense(16, activation='relu', name='enc_dense1')(inputs)
    x = keras.layers.BatchNormalization(name='enc_bn1')(x)
    encoded = keras.layers.Dense(encoding_dim, activation='relu', name='bottleneck')(x)

    # --- Decoder ---
    x = keras.layers.Dense(16, activation='relu', name='dec_dense1')(encoded)
    x = keras.layers.BatchNormalization(name='dec_bn1')(x)
    decoded = keras.layers.Dense(input_dim, activation='linear', name='sensor_output')(x)

    model = keras.Model(inputs=inputs, outputs=decoded, name='SensorAutoencoder')
    return model


sensor_model = build_sensor_autoencoder(INPUT_DIM, ENCODING_DIM)
sensor_model.summary()

## 6. Training

In [ ]:
sensor_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

callbacks = [
    # Hentikan training kalau val_loss tidak membaik selama 15 epoch
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,  # kembalikan bobot terbaik setelah stop
        verbose=1
    ),
    # Turunkan learning rate otomatis kalau val_loss stagnan
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-6,
        verbose=1
    )
]

print("Memulai training...")
history = sensor_model.fit(
    X_train, X_train,                    # input = target (ini ciri khas autoencoder)
    validation_data=(X_val, X_val),
    epochs=200,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

final_epoch = len(history.history['loss'])
print(f"\nTraining selesai di epoch : {final_epoch}")
print(f"Final train loss          : {history.history['loss'][-1]:.6f}")
print(f"Final val loss            : {history.history['val_loss'][-1]:.6f}")

## 7. Visualisasi Training

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(history.history['loss'],     label='Train Loss', color='steelblue', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss',   color='tomato',    linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE curve
axes[1].plot(history.history['mae'],     label='Train MAE', color='steelblue', linewidth=2)
axes[1].plot(history.history['val_mae'], label='Val MAE',   color='tomato',    linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].set_title('Training & Validation MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Sensor Autoencoder — Kurva Training', fontsize=14)
plt.tight_layout()
plt.savefig('sensor_training_loss.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Hitung Reconstruction Error & Threshold

Threshold dihitung dari **persentil ke-95** reconstruction error pada data training.  
Artinya: error di atas threshold = anomali (kira-kira 5% teratas dianggap tidak normal).

In [ ]:
# Hitung reconstruction error di data TRAINING (bukan val)
X_train_recon = sensor_model.predict(X_train, verbose=0)
train_errors  = np.mean((X_train - X_train_recon) ** 2, axis=1)

# Hitung juga untuk val (untuk cek konsistensi)
X_val_recon = sensor_model.predict(X_val, verbose=0)
val_errors  = np.mean((X_val - X_val_recon) ** 2, axis=1)

print("=== Reconstruction Error (Training Data) ===")
print(f"  Min  : {train_errors.min():.6f}")
print(f"  Max  : {train_errors.max():.6f}")
print(f"  Mean : {train_errors.mean():.6f}")
print(f"  Std  : {train_errors.std():.6f}")
print(f"  P90  : {np.percentile(train_errors, 90):.6f}")
print(f"  P95  : {np.percentile(train_errors, 95):.6f}")
print(f"  P99  : {np.percentile(train_errors, 99):.6f}")
print()
print("=== Reconstruction Error (Validation Data) ===")
print(f"  Mean : {val_errors.mean():.6f}")
print(f"  P95  : {np.percentile(val_errors, 95):.6f}")

# Threshold = P95 dari training error
sensor_threshold = float(np.percentile(train_errors, 95))
print(f"\n>>> Threshold yang dipakai (P95 train): {sensor_threshold:.6f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribusi reconstruction error
axes[0].hist(train_errors, bins=50, alpha=0.7, color='steelblue',
             edgecolor='white', label='Train error')
axes[0].hist(val_errors,   bins=50, alpha=0.5, color='seagreen',
             edgecolor='white', label='Val error')
axes[0].axvline(x=sensor_threshold, color='red', linestyle='--',
                linewidth=2, label=f'Threshold P95 = {sensor_threshold:.4f}')
axes[0].set_xlabel('Reconstruction Error (MSE)')
axes[0].set_ylabel('Frekuensi')
axes[0].set_title('Distribusi Reconstruction Error')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Reconstruction error per sampel (train)
axes[1].scatter(range(len(train_errors)), np.sort(train_errors),
                s=5, alpha=0.5, color='steelblue', label='Train')
axes[1].axhline(y=sensor_threshold, color='red', linestyle='--',
                linewidth=2, label=f'Threshold = {sensor_threshold:.4f}')
axes[1].set_xlabel('Sampel (diurutkan)')
axes[1].set_ylabel('Reconstruction Error')
axes[1].set_title('Error Terurut — Train')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Sensor Autoencoder — Analisis Reconstruction Error', fontsize=14)
plt.tight_layout()
plt.savefig('sensor_error_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Berapa % training data dianggap anomali (harusnya ~5%)
anomaly_rate_train = (train_errors > sensor_threshold).mean() * 100
anomaly_rate_val   = (val_errors   > sensor_threshold).mean() * 100
print(f"Anomaly rate training : {anomaly_rate_train:.1f}% (target: ~5%)")
print(f"Anomaly rate validasi : {anomaly_rate_val:.1f}%")

## 9. Evaluasi: Cek Apakah Model Masuk Akal

Karena dataset punya kolom `target`, kita bisa cek apakah reconstruction error berkorelasi dengan kondisi koloni.  
Ini **bukan** bagian dari training — hanya untuk validasi bahwa model menangkap sesuatu yang bermakna.

In [ ]:
# Hitung error untuk seluruh dataset
X_all_norm  = sensor_scaler.transform(X_sensor)
X_all_recon = sensor_model.predict(X_all_norm, verbose=0)
all_errors  = np.mean((X_all_norm - X_all_recon) ** 2, axis=1)

df_eval = df[SENSOR_FEATURES + ['target']].copy()
df_eval['reconstruction_error'] = all_errors
df_eval['predicted_anomaly']    = all_errors > sensor_threshold

print("=== Rata-rata Reconstruction Error per Kelas Target ===")
print(df_eval.groupby('target')['reconstruction_error'].agg(['mean', 'std', 'count']).round(6))

In [ ]:
# Boxplot error per target class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

target_groups = [df_eval[df_eval['target'] == t]['reconstruction_error'].values
                 for t in sorted(df_eval['target'].unique())]

axes[0].boxplot(target_groups,
                labels=[f'Target {t}' for t in sorted(df_eval['target'].unique())],
                patch_artist=True)
axes[0].axhline(y=sensor_threshold, color='red', linestyle='--',
                linewidth=2, label=f'Threshold = {sensor_threshold:.4f}')
axes[0].set_ylabel('Reconstruction Error')
axes[0].set_title('Reconstruction Error per Kelas Target')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Anomaly rate per target
anomaly_by_target = df_eval.groupby('target')['predicted_anomaly'].mean() * 100
bars = axes[1].bar(anomaly_by_target.index.astype(str),
                   anomaly_by_target.values,
                   color='steelblue', alpha=0.8, edgecolor='white')
axes[1].axhline(y=5, color='red', linestyle='--', linewidth=2, label='5% baseline')
axes[1].set_xlabel('Target Class')
axes[1].set_ylabel('Anomaly Rate (%)')
axes[1].set_title('% Sampel Dianggap Anomali per Kelas Target')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, anomaly_by_target.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=10)

plt.suptitle('Sensor Autoencoder — Evaluasi terhadap Label Target', fontsize=14)
plt.tight_layout()
plt.savefig('sensor_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Visualisasi bottleneck space (2D)
# Buat encoder-only model untuk ekstrak representasi bottleneck
encoder_model = keras.Model(
    inputs=sensor_model.input,
    outputs=sensor_model.get_layer('bottleneck').output
)

bottleneck_repr = encoder_model.predict(X_all_norm, verbose=0)

plt.figure(figsize=(8, 6))
targets = df['target'].values
scatter = plt.scatter(
    bottleneck_repr[:, 0],
    bottleneck_repr[:, 1],
    c=targets, cmap='tab10', alpha=0.6, s=20
)
plt.colorbar(scatter, label='Target Class')
plt.xlabel('Dimensi Bottleneck 1')
plt.ylabel('Dimensi Bottleneck 2')
plt.title('Representasi Bottleneck (2D) — Diwarnai per Kelas Target')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('sensor_bottleneck.png', dpi=150, bbox_inches='tight')
plt.show()

print("Kalau ada pemisahan visual antar warna → model menangkap pola yang bermakna.")

## 10. Simpan Semua Artefak

In [ ]:
# 1. Simpan model Keras
sensor_model.save('models/sensor_autoencoder.keras')
print("✅ Model     : models/sensor_autoencoder.keras")

# 2. Simpan scaler
with open('models/sensor_scaler.pkl', 'wb') as f:
    pickle.dump(sensor_scaler, f)
print("✅ Scaler    : models/sensor_scaler.pkl")

# 3. Simpan threshold
np.save('models/sensor_threshold.npy', np.array(sensor_threshold, dtype=np.float32))
print("✅ Threshold : models/sensor_threshold.npy")

# 4. Simpan metadata model (untuk referensi tim)
model_info = {
    'input_features'       : SENSOR_FEATURES,
    'input_dim'            : INPUT_DIM,
    'encoding_dim'         : ENCODING_DIM,
    'threshold_percentile' : 95,
    'threshold_value'      : sensor_threshold,
    'training_samples'     : int(X_train.shape[0]),
    'val_samples'          : int(X_val.shape[0]),
    'final_train_loss'     : float(history.history['loss'][-1]),
    'final_val_loss'       : float(history.history['val_loss'][-1]),
    'epochs_trained'       : len(history.history['loss']),
    'tensorflow_version'   : tf.__version__,
    'scaler_mean'          : sensor_scaler.mean_.tolist(),
    'scaler_std'           : sensor_scaler.scale_.tolist(),
}
with open('models/sensor_model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)
print("✅ Info      : models/sensor_model_info.json")

print()
print("--- Ringkasan ---")
for k, v in model_info.items():
    print(f"  {k:25s}: {v}")

## 11. Verifikasi — Simulasi Inferensi dari ESP32

Ini mensimulasikan apa yang server lakukan saat menerima data dari perangkat IoT.

In [ ]:
# Load ulang semua artefak untuk memastikan bisa di-load dengan benar
loaded_model     = tf.keras.models.load_model('models/sensor_autoencoder.keras')
with open('models/sensor_scaler.pkl', 'rb') as f:
    loaded_scaler = pickle.load(f)
loaded_threshold = float(np.load('models/sensor_threshold.npy'))

print("=== Verifikasi Load Artefak ===")
print(f"  Model load    : OK")
print(f"  Scaler load   : OK")
print(f"  Threshold     : {loaded_threshold:.6f}")
print()

# -------------------------------------------------------
# Simulasi: data masuk dari ESP32 dalam format JSON
# {"temperature": 34.5, "humidity": 58.3, "pressure": 1013.2}
# -------------------------------------------------------

def infer_from_esp32(temperature: float, humidity: float, pressure: float) -> dict:
    """
    Fungsi inferensi yang sama persis dengan yang akan berjalan di server.
    Input: nilai sensor mentah dari ESP32
    Output: dict berisi error dan status anomali
    """
    # 1. Susun array sesuai urutan fitur saat training
    sensor_raw = np.array([[temperature, humidity, pressure]])

    # 2. Normalisasi dengan scaler yang sudah di-fit saat training
    sensor_norm = loaded_scaler.transform(sensor_raw)

    # 3. Rekonstruksi
    reconstruction = loaded_model.predict(sensor_norm, verbose=0)

    # 4. Hitung reconstruction error (MSE)
    error = float(np.mean((sensor_norm - reconstruction) ** 2))

    return {
        'sensor_error'    : error,
        'sensor_threshold': loaded_threshold,
        'sensor_anomaly'  : error > loaded_threshold
    }


# Test 1: data yang terlihat normal
print("=== Test 1: Kondisi Normal ===")
result1 = infer_from_esp32(temperature=28.0, humidity=45.0, pressure=1009.0)
print(f"  Input         : temp=28.0°C, hum=45.0%, pres=1009.0 hPa")
print(f"  Sensor error  : {result1['sensor_error']:.6f}")
print(f"  Threshold     : {result1['sensor_threshold']:.6f}")
print(f"  Anomali?      : {'YA ⚠️' if result1['sensor_anomaly'] else 'TIDAK ✅'}")

print()

# Test 2: data dengan suhu sangat tinggi (potensi anomali)
print("=== Test 2: Suhu Ekstrem ===")
result2 = infer_from_esp32(temperature=55.0, humidity=90.0, pressure=1005.0)
print(f"  Input         : temp=55.0°C, hum=90.0%, pres=1005.0 hPa")
print(f"  Sensor error  : {result2['sensor_error']:.6f}")
print(f"  Threshold     : {result2['sensor_threshold']:.6f}")
print(f"  Anomali?      : {'YA ⚠️' if result2['sensor_anomaly'] else 'TIDAK ✅'}")

In [ ]:
print("=" * 50)
print("SEMUA ARTEFAK VALID — SIAP DI-PUSH KE GITHUB")
print("=" * 50)
print()
print("File yang perlu di-push:")
print("  models/sensor_autoencoder.keras")
print("  models/sensor_scaler.pkl")
print("  models/sensor_threshold.npy")
print("  models/sensor_model_info.json")
print()
print("Kontrak teknis untuk tim:")
print(f"  Input fitur  : {SENSOR_FEATURES}")
print(f"  Input dim    : {INPUT_DIM}")
print(f"  Threshold    : {sensor_threshold:.6f}")
print(f"  Format model : .keras (TensorFlow/Keras)")